In [48]:
import numpy as np
import torch
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from torch_geometric.datasets import MoleculeNet
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use("Agg")  # backend non interactif : sauvegarde sans affichage
import matplotlib.pyplot as plt
from pathlib import Path

TASK_NAMES = ["NR-AR", "NR-AR-LBD", "NR-AhR", "NR-ER", "NR-ER-LBD",
              "NR-PPAR-gamma", "NR-Aromatase", "SR-ARE", "SR-ATAD5",
              "SR-HSE", "SR-MMP", "SR-p53"]

FIG_DIR = Path("results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Generateur Morgan instancie UNE fois (nouvelle API RDKit, plus de deprecation)
MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


# ------------------------------------------------------------- chargement dataset

def load_tox21(root: str = "data/") -> MoleculeNet:
    """Tox21 via PyG : telecharge + cache au premier appel (dossier data/, ignore par git).
    Chaque molecule : x (atomes), edge_index (liaisons), y (12 taches, NaN = non teste),
    smiles."""
    dataset = MoleculeNet(root=root, name="Tox21")
    print(f"Tox21 : {len(dataset)} molecules")
    return dataset


# ------------------------------------------------------------------ scaffold split

def murcko_scaffold_smiles(data) -> str:
    """SMILES canonique du scaffold Bemis-Murcko : la clef de regroupement."""
    mol = Chem.MolFromSmiles(data.smiles)
    if mol is None:
        return ""
    return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))


def scaffold_split(dataset, frac_train=0.8, frac_val=0.1, frac_test=0.1,
                   seed: int = 0):
    """Scaffold split : toutes les molecules d'un meme scaffold dans le MEME split
    -> pas de fuite structurelle (contrairement au split aleatoire).
    Groupes tries par taille decroissante, remplissage train -> val -> test,
    ordre melange par seed -> split deterministe."""
    assert abs(frac_train + frac_val + frac_test - 1.0) < 1e-6
    rng = torch.Generator().manual_seed(seed)

    scaffold_to_idx = {}
    for i in range(len(dataset)):
        scaffold_to_idx.setdefault(murcko_scaffold_smiles(dataset[i]), []).append(i)

    # Le seed doit melanger l'ordre DES GROUPES (pas leur interieur) :
    # permuter l'interieur d'un groupe ne change pas le split de ses membres,
    # d'ou des results rigoureusement identiques pour tous les seeds (bug fixe).
    groups = sorted(scaffold_to_idx.values(), key=len, reverse=True)
    order = torch.randperm(len(groups), generator=rng).tolist()
    groups = [groups[k] for k in order]

    n = len(dataset)
    train, val, test = [], [], []
    for group in groups:
        for idx in group:
            if len(train) < frac_train * n:
                train.append(idx)
            elif len(val) < frac_val * n:
                val.append(idx)
            else:
                test.append(idx)
    return train, val, test


# ---------------------------------------------------------------- featurisation

def smiles_to_fingerprint(smiles: str, radius: int = 2, n_bits: int = 2048) -> np.ndarray:
    """ECFP4 : SMILES -> vecteur binaire (n_bits,). Sparse, sans fit -> pas de fuite."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"SMILES invalide : {smiles!r}")
    fp = MORGAN_GEN.GetFingerprint(mol)
    return np.asarray(fp, dtype=np.float32)


def build_feature_matrix(dataset) -> np.ndarray:
    """Tout le dataset -> X (n_molecules, n_bits). Une seule fois, reutilise partout."""
    X = np.stack([smiles_to_fingerprint(d.smiles) for d in dataset])
    print(f"Features : X {X.shape}, densite {X.mean():.4f}")
    return X


def stack_labels(dataset) -> np.ndarray:
    """y de PyG (n, 1, 12) -> Y (n, 12) avec NaN conserves."""
    return torch.stack([d.y for d in dataset]).squeeze(1).numpy()


# --------------------------------------------------------------------- entrainement

def train_svm_per_task(X: np.ndarray,
                       Y: np.ndarray,
                       idx_train: list,
                       C: float = 1.0):
    """Un LinearSVC par tache, sur les lignes annotees uniquement.
    class_weight='balanced' compense le desequilibre mesure dans l'EDA.
    Decision_function servira pour l'AUC -> pas de probability=True (Platt couteux)."""
    models = []
    for t in range(Y.shape[1]):
        mask_tr = ~np.isnan(Y[idx_train, t])
        Xtr = X[idx_train][mask_tr]
        ytr = Y[idx_train, t][mask_tr].astype(int)
        clf = LinearSVC(C=C, class_weight="balanced", max_iter=10_000)
        clf.fit(Xtr, ytr)
        models.append(clf)
    return models


# ---------------------------------------------------------------------- evaluation

def evaluate_svm(models, X: np.ndarray, Y: np.ndarray, idx) -> dict:
    """AUC par tache sur les lignes annotees du split, + moyenne.
    AUC = metrique de ranking : les marges signees suffisent."""
    aucs = []
    for t, clf in enumerate(models):
        mask = ~np.isnan(Y[idx, t])
        if mask.sum() < 2 or len(np.unique(Y[idx, t][mask])) < 2:
            aucs.append(np.nan)
            continue
        scores = clf.decision_function(X[idx][mask])
        aucs.append(roc_auc_score(Y[idx, t][mask].astype(int), scores))
    aucs = np.array(aucs)
    return {"auc_per_task": aucs, "auc_mean": np.nanmean(aucs)}



def train_svm_for_task(X, Y, idx_train, t, C):
    mask = ~np.isnan(Y[idx_train, t])
    clf = LinearSVC(C=C, class_weight="balanced", max_iter=10_000)
    clf.fit(X[idx_train][mask], Y[idx_train, t][mask].astype(int))
    return clf


def auc_for_task(clf, X, Y, idx_val, t) -> float:
    mask = ~np.isnan(Y[idx_val, t])
    if mask.sum() < 2 or len(np.unique(Y[idx_val, t][mask])) < 2:
        return np.nan
    return roc_auc_score(Y[idx_val, t][mask].astype(int),
                         clf.decision_function(X[idx_val][mask]))


# ------------------------------------------------------------------------- pipeline

def plot_auc_per_task(auc_matrix: np.ndarray, out: Path = FIG_DIR / "svm_auc_per_task.png"):
    """Barres horizontales : AUC par tache (moyenne +/- std sur les seeds).
    Sauvegarde dans results/figures (versionne) + retourne la figure."""
    aucs = np.nanmean(auc_matrix, axis=0)
    stds = np.nanstd(auc_matrix, axis=0)
    order = np.argsort(aucs)  # triees de la pire a la meilleure

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(np.array(TASK_NAMES)[order], aucs[order],
            xerr=stds[order], color="#4C72B0", alpha=0.85)
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="Hasard (0.5)")
    ax.set_xlim(0.4, 0.85)
    ax.set_xlabel("AUC ROC (test, scaffold split)")
    ax.set_title("Baseline SVM lineaire + ECFP4 - AUC par tache")
    ax.legend()
    fig.tight_layout()
 #   fig.savefig(out, dpi=150)
    print(f"Figure sauvegardee : {out}")
    return fig


# ------------------------------------------------------------------------- pipeline

def run(seed_list=(0, 1, 2,3,4,5,6,7,8,9,10,11,20,45,67), C: float = 1.0):
    dataset = load_tox21()
    X = build_feature_matrix(dataset)
    Y = stack_labels(dataset)

    all_means, all_aucs = [], []
    for seed in seed_list:
        train, val, test = scaffold_split(dataset, seed=seed)
        models = train_svm_per_task(X, Y, train, C=C)
        res = evaluate_svm(models, X, Y, test)
        print(f"seed={seed} | AUC moyenne test = {res['auc_mean']:.4f}")
        print("  par tache :", np.round(res["auc_per_task"], 3))
        all_means.append(res["auc_mean"])
        all_aucs.append(res["auc_per_task"])

    m = np.array(all_means)
    print(f"\nSVM lineaire : AUC moyenne = {m.mean():.4f} +/- {m.std():.4f}")

    # Tableau console lisible
    auc_matrix = np.stack(all_aucs)                      # (n_seeds, 12)
    print("\n" + "-" * 50)
    for name, a, s in zip(TASK_NAMES, np.nanmean(auc_matrix, 0), np.nanstd(auc_matrix, 0)):
        print(f"  {name:<14} {a:.3f} +/- {s:.3f}")
    print("-" * 50)

    plot_auc_per_task(auc_matrix)
    np.savetxt(FIG_DIR.parent / "svm_auc_per_task.csv",
               auc_matrix, delimiter=",", header=",".join(TASK_NAMES), comments="")
    return m.mean()


#if __name__ == "__main__":
#   run()

In [55]:
import math
import random
from tqdm import tqdm
from typing import List, Tuple

import numpy as np

def op_score(c : float):
    auc_mean=run(C=c)
    return auc_mean

def _neighbor(c: float) -> float:
    # Génère la valeur et l'extrait directement sous forme de float
    return abs(float(np.random.normal(c, c / 4)))
def _run_asa(
        initial_solution: float,
        niter: int = 50,
        temp: float = 1.0,
        cooling: float = 0.99,
        target: float = 0.44,
    ) -> Tuple[float, float,List[float]]:
        """Exécute l'algorithme ASA à partir d'une solution initiale."""
        cur = initial_solution
        Sol=[cur]
        cur_score = best_score = op_score(cur)
        best = cur
        acc, it = 0, 0

        for _ in tqdm(range(niter)):
            new = _neighbor(cur)
            new_score = op_score(new)
            if new_score == 0:
                return new, 0.0
            delta = new_score - cur_score
            if delta < 0 or random.random() < math.exp(-delta / temp):
                cur, cur_score = new, new_score
                Sol.append(cur)
                acc += 1
                if cur_score < best_score:
                    best, best_score = cur, cur_score
            it += 1
            if it % 10 == 0:
                rate = acc / it if it > 0 else 0.0
                temp *= math.exp((rate - target) / target)
                acc, it = 0, 0
            temp *= cooling
        return best, best_score,Sol

In [ ]:
_run_asa(1)

Tox21 : 7823 molecules


[21:18:26] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146


[21:18:48] WARNING: not removing hydrogen atom without neighbors


seed=0 | AUC moyenne test = 0.6885
  par tache : [0.637 0.665 0.76  0.705 0.585 0.645 0.731 0.64  0.698 0.688 0.794 0.714]


...

  8%|▊         | 4/50 [17:34<3:18:56, 259.50s/it]

Tox21 : 7823 molecules


[21:39:38] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146


[21:39:54] WARNING: not removing hydrogen atom without neighbors
